In [2]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)


In [3]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [4]:
from minsearch import AppendableIndex

index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [5]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
    )

    return results


search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [6]:
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.runners import DisplayingRunnerCallback
from toyaikit.tools import Tools

In [7]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [8]:
instructions = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.

If you want to look up the answer, explain why before making the call
""".strip()

In [9]:
chat_interface = IPythonChatInterface()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient()
)


In [10]:
chat_interface = IPythonChatInterface()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient()
)



In [11]:
callback = DisplayingRunnerCallback(chat_interface)

question = 'how do I install kafka'
loop_result = runner.loop(prompt=question, callback=callback)


In [12]:
loop_result.cost

CostInfo(input_cost=0.0001866, output_cost=0.0002904, total_cost=0.000477)

# OpenAO Agents SDK

In [14]:
import agents
agents.__version__

'0.4.1'

In [24]:
from agents import Agent, function_tool, Runner

In [21]:
import requests
from requests.exceptions import RequestException, HTTPError, Timeout, ConnectionError

def get_page_content(url: str) -> str:
    """
    Fetches the textual content of a web page using the Jina Reader API.

    This function sends a GET request to the Jina Reader endpoint, which retrieves
    and returns the clean text content of the specified URL (useful for extracting
    article or webpage text without boilerplate HTML).

    Args:
        url (str): The URL of the web page to fetch.

    Returns:
        str: The decoded UTF-8 content of the page retrieved through the Jina Reader API.
             Returns an empty string if the request fails.

    Raises:
        ValueError: If the provided URL is invalid or empty.
    """
    if not url or not isinstance(url, str):
        raise ValueError("A valid non-empty URL string must be provided.")

    reader_url_prefix = "https://r.jina.ai/"
    request_url = reader_url_prefix + url

    try:
        response = requests.get(request_url, timeout=10)
        response.raise_for_status()
        return response.content.decode("utf-8")

    except Timeout:
        print(f"Request timed out while trying to fetch: {url}")
    except ConnectionError:
        print(f"Connection error occurred while trying to fetch: {url}")
    except HTTPError as e:
        print(f"HTTP error {e.response.status_code} occurred for: {url}")
    except RequestException as e:
        print(f"An error occurred while fetching the URL: {url}\nDetails: {e}")

    # Return empty string if an error occurred
    return ""


In [22]:
content = get_page_content('https://datatalks.club')
print(content)

Title: Welcome to DataTalks.Club

URL Source: https://datatalks.club/

Published Time: Tue, 21 Oct 2025 12:52:05 GMT

Markdown Content:
Welcome to DataTalks.Club


AI Dev Tools Zoomcamp: Learn AI-powered coding assistants and agents[Register here!](https://airtable.com/appJRFiWKHBgmEt70/shrpw7rk55Ewr1jCG)

DataTalks.Club
--------------

[Articles](https://datatalks.club/articles.html)[Slack](https://datatalks.club/slack.html)[Events](https://datatalks.club/events.html)[Podcast](https://datatalks.club/podcast.html)[Books](https://datatalks.club/books.html)[Courses](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html)

* * *

The place to talk about data

Global online community of data science professionals, ML engineers, and AI practitioners
-----------------------------------------------------------------------------------------

Subscribe to our weekly newsletter and join our Slack.

 We'll keep you informed about everything happening in the Club.

Email 


In [36]:
web_agent = Agent(
    name ='web_agent',
    instructions = 'you are helpful assistant',
    model = 'gpt-4o-mini', 
    tools = [function_tool(get_page_content)]
) 

In [26]:
# The runner enables the loop  

from agents import Runner
runner = Runner()


In [31]:
question = "what is this page about? https://openai.github.io/openai-agents-python/"

In [37]:
results = await runner.run(web_agent, input = question )


/var/folders/z0/vchk24hj0y707vpcv211pl1w0000gn/T/ipykernel_29363/3551963148.py:1: RuntimeWarning: coroutine 'Runner.run' was never awaited
  results = await runner.run(web_agent, input = question )


In [39]:
print(results.final_output)

The page you referred to is the documentation for the **OpenAI Agents SDK**, which enables developers to build AI applications with agent-like behaviors. Here are the key points:

### Overview
- **OpenAI Agents SDK** provides a streamlined way to create agentic AI apps using a lightweight package.
- It is a production-ready evolution of earlier experimentation with agents.

### Key Features
- **Agents**: Large Language Models (LLMs) with specific instructions and tools.
- **Handoffs**: Allows agents to delegate tasks to other agents.
- **Guardrails**: Ensures validation of agent inputs and outputs.
- **Sessions**: Manages conversation history automatically.

### Benefits
- The SDK aims to be easy to learn while providing sufficient features for practical use.
- It works seamlessly with Python, allowing users to utilize language features easily.
- Built-in features for debugging, visualization, and monitoring workflows.

### Installation
To install, use:
```bash
pip install openai-agent

In [45]:
# This reflects the conversations in the agent
items= results.new_items

In [48]:
items[-1].raw_item

ResponseOutputMessage(id='msg_0f866ba1239232960068f92799928c81949e98a21f0286dd0d', content=[ResponseOutputText(annotations=[], text='The page you referred to is the documentation for the **OpenAI Agents SDK**, which enables developers to build AI applications with agent-like behaviors. Here are the key points:\n\n### Overview\n- **OpenAI Agents SDK** provides a streamlined way to create agentic AI apps using a lightweight package.\n- It is a production-ready evolution of earlier experimentation with agents.\n\n### Key Features\n- **Agents**: Large Language Models (LLMs) with specific instructions and tools.\n- **Handoffs**: Allows agents to delegate tasks to other agents.\n- **Guardrails**: Ensures validation of agent inputs and outputs.\n- **Sessions**: Manages conversation history automatically.\n\n### Benefits\n- The SDK aims to be easy to learn while providing sufficient features for practical use.\n- It works seamlessly with Python, allowing users to utilize language features easi

# Youtube videos

In [57]:
from typing import List, Dict, Any, Iterable, Optional
import re
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled,
    NoTranscriptFound,
)


def format_timestamp(seconds: float) -> str:
    """Convert seconds to H:MM:SS if > 1 hour, else M:SS."""
    total_seconds = int(seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours}:{minutes:02}:{secs:02}" if hours > 0 else f"{minutes}:{secs:02}"


def make_subtitles(transcript: Iterable[Dict[str, Any]]) -> str:
    """Convert transcript entries to plain text with timestamps."""
    lines: List[str] = []
    for entry in transcript:
        start = entry.get("start", 0.0)
        text = (entry.get("text") or "").replace("\n", " ").strip()
        if not text:
            continue
        ts = format_timestamp(start)
        lines.append(f"{ts} {text}")
    return "\n".join(lines)


def _extract_video_id(video_id_or_url: str) -> str:
    """Extract the 11-character video ID from a YouTube URL or return it unchanged."""
    patterns = [
        r"v=([A-Za-z0-9_-]{11})",
        r"youtu\.be/([A-Za-z0-9_-]{11})",
        r"embed/([A-Za-z0-9_-]{11})",
    ]
    for pat in patterns:
        m = re.search(pat, video_id_or_url)
        if m:
            return m.group(1)
    return video_id_or_url


def fetch_transcript_raw(
    video_id_or_url: str,
    preferred_langs: Iterable[str] = ("en", "es"),
    translation_target: Optional[str] = "en",
) -> List[Dict[str, Any]]:
    """
    Fetch a YouTube transcript with fallback for language and translation.

    Strategy:
    1) Try to get a transcript in preferred languages.
    2) If not found, translate a translatable transcript to target language.
    3) Fallback to first available transcript if none match.
    """
    video_id = _extract_video_id(video_id_or_url)

    try:
        # ✅ Proper usage — call the function as a static method on the class
        transcripts = YouTubeTranscriptApi.list_transcripts(video_id)

        # Try preferred languages first
        for lang in preferred_langs:
            try:
                return transcripts.find_transcript([lang]).fetch()
            except NoTranscriptFound:
                continue

        # Try translatable transcripts
        if translation_target:
            for t in transcripts:
                if getattr(t, "is_translatable", False):
                    try:
                        return t.translate(translation_target).fetch()
                    except Exception:
                        continue

        # Fallback to any transcript available
        for t in transcripts:
            try:
                return t.fetch()
            except Exception:
                continue

        raise NoTranscriptFound(f"No usable transcript found for {video_id}.")

    except TranscriptsDisabled:
        raise TranscriptsDisabled(f"Transcripts are disabled for video {video_id}.")
    except NoTranscriptFound:
        raise NoTranscriptFound(
            f"No transcript found for {video_id}. Tried {preferred_langs}."
        )


def fetch_transcript_text(
    video_id_or_url: str,
    preferred_langs: Iterable[str] = ("en", "es"),
    translation_target: Optional[str] = "en",
) -> str:
    """Return transcript text with timestamps, using same fallback logic."""
    transcript = fetch_transcript_raw(
        video_id_or_url,
        preferred_langs=preferred_langs,
        translation_target=translation_target,
    )
    return make_subtitles(transcript)



In [60]:
video_url = "https://www.youtube.com/watch?v=0IU-nb08HR4"

try:
    transcript_text = fetch_transcript_text(video_url)
    print(transcript_text[:600])
except Exception as e:
    print(f"Error: {e}")

Error: type object 'YouTubeTranscriptApi' has no attribute 'list_transcripts'


# Pydantic AI
